# Model Interpretation & Risk Output

This notebook converts model results into business-ready Early Warning System outputs.

It creates:

- final model decision per `product_id`, `region`, `year_month`
- `risk_score` and `high_risk_probability`
- risk label: `Low Risk`, `Medium Risk`, `High Risk`
- most important risk drivers per product-month
- top-risk products table
- recall vs false-alarm trade-off chart
- dashboard-ready CSV for Power BI / Tableau

The notebook retrains a compact NumPy Random Forest on the feature-engineered ML-ready files so it can generate row-level probabilities for train, validation and test splits. The local risk drivers are rule-based and business-readable, which is often better for dashboard users than raw model internals.


## Interpretation Strategy

The model produces probabilities, but business users need decisions and reasons. Therefore this notebook converts `high_risk_probability` into a `risk_score`, applies a selected threshold, and attaches readable risk drivers such as sales drop, customer drop, volatility, supply pressure, margin risk, and weak market context.


In [10]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
TARGET_COL = "next_month_risk_label"
LABEL_NAMES = {0: "Low Risk", 1: "Medium Risk", 2: "High Risk"}

INPUT_DIR = Path("../data/feature_engineering")
OUTPUT_DIR = Path("../data/ml_interpretation_risk_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
# Load ML-ready matrices, IDs, and human-readable engineered dataset
def load_inputs():
    data = {}
    for split in ["train", "valid", "test"]:
        data[f"{split}_ml"] = pd.read_csv(INPUT_DIR / f"{split}_feature_engineered_ml_ready.csv")
        data[f"{split}_ids"] = pd.read_csv(INPUT_DIR / f"{split}_feature_engineered_ids.csv")
        data[f"{split}_ids"]["year_month"] = pd.to_datetime(data[f"{split}_ids"]["year_month"])

    human = pd.read_csv(INPUT_DIR / "feature_engineered_modeling_dataset.csv")
    human["year_month"] = pd.to_datetime(human["year_month"])
    return data, human


def split_xy(df):
    X = df.drop(columns=[TARGET_COL]).copy()
    y = df[TARGET_COL].astype(int).to_numpy()
    return X, y

def high_risk_metrics(y_true, y_pred):
    y_true_high = y_true == 2
    y_pred_high = y_pred == 2
    tp = int(np.sum(y_true_high & y_pred_high))
    fp = int(np.sum(~y_true_high & y_pred_high))
    fn = int(np.sum(y_true_high & ~y_pred_high))
    tn = int(np.sum(~y_true_high & ~y_pred_high))
    return {
        "high_risk_recall": tp / (tp + fn) if (tp + fn) else 0,
        "high_risk_precision": tp / (tp + fp) if (tp + fp) else 0,
        "false_alarm_rate": fp / (fp + tn) if (fp + tn) else 0,
        "alert_rate": (tp + fp) / len(y_true),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn}

In [12]:
def confusion_matrix_np(y_true, y_pred):
    cm = np.zeros((3, 3), dtype=int)
    for actual, pred in zip(y_true, y_pred):
        cm[int(actual), int(pred)] += 1
    return cm

def classification_report_np(y_true, y_pred):
    cm = confusion_matrix_np(y_true, y_pred)
    rows = []
    for label in [0, 1, 2]:
        tp = cm[label, label]
        fp = cm[:, label].sum() - tp
        fn = cm[label, :].sum() - tp
        support = cm[label, :].sum()
        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
        rows.append({
            "label": label,
            "label_name": LABEL_NAMES[label],
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": int(support)})
    report = pd.DataFrame(rows)
    metrics = {
        "accuracy": np.trace(cm) / cm.sum(),
        "macro_f1": report["f1_score"].mean(),
        "weighted_f1": np.average(report["f1_score"], weights=report["support"]),
        **high_risk_metrics(y_true, y_pred)}
    return report, cm, metrics

In [13]:
# Compact multiclass tree used inside the interpretation Random Forest
class FastRandomTree:
    def __init__(self, max_depth=7, min_leaf=90, max_features=12, n_thresholds=7, class_weight=None, seed=42):
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.max_features = max_features
        self.n_thresholds = n_thresholds
        self.class_weight = class_weight or {0: 1.0, 1: 1.0, 2: 1.0}
        self.rng = np.random.default_rng(seed)

    def _counts(self, y): return np.array([np.sum(y == c) * self.class_weight.get(c, 1.0) for c in [0, 1, 2]], dtype=float)

    def _proba(self, y):
        counts = self._counts(y) + 1e-6
        return counts / counts.sum()

    def _gini_counts(self, counts):
        p = counts / counts.sum()
        return 1 - np.sum(p ** 2)

    def _gini(self, y): return self._gini_counts(self._counts(y))

    def fit(self, X, y, n_features):
        self.feature_importances_ = np.zeros(n_features)
        self.tree_ = self._build(X, y, 0)
        total = self.feature_importances_.sum()
        if total:
            self.feature_importances_ /= total
        return self

    def _best_split(self, X, y):
        parent_gini = self._gini(y)
        best = (None, None, None, 0.0)
        features = self.rng.choice(X.shape[1], size=min(self.max_features, X.shape[1]), replace=False)
        for f in features:
            x = X[:, f]
            thresholds = np.unique(np.quantile(x[np.isfinite(x)], np.linspace(0.15, 0.85, self.n_thresholds)))
            for t in thresholds:
                left = x <= t
                if left.sum() < self.min_leaf or (~left).sum() < self.min_leaf:
                    continue
                lc = self._counts(y[left])
                rc = self._counts(y[~left])
                total = lc.sum() + rc.sum()
                child_gini = (lc.sum() / total) * self._gini_counts(lc) + (rc.sum() / total) * self._gini_counts(rc)
                gain = parent_gini - child_gini
                if gain > best[3]:
                    best = (f, t, left, gain)
        return best

    def _build(self, X, y, depth):
        node = {"proba": self._proba(y), "class": int(np.argmax(self._proba(y)))}
        if depth >= self.max_depth or len(y) < self.min_leaf * 3 or len(np.unique(y)) == 1:
            return node
        f, t, left, gain = self._best_split(X, y)
        if f is None or gain <= 1e-10:
            return node
        self.feature_importances_[f] += gain * len(y)
        node.update({
            "feature": int(f),
            "threshold": float(t),
            "left": self._build(X[left], y[left], depth + 1),
            "right": self._build(X[~left], y[~left], depth + 1)})
        return node

    def _predict_one(self, row):
        node = self.tree_
        while "feature" in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        return node["proba"]

    def predict_proba(self, X):
        return np.vstack([self._predict_one(row) for row in X])

# Small NumPy Random Forest for row-level probabilities
class FastRandomForest:

    def __init__(self, n_estimators=24, max_depth=7, row_subsample=0.72, class_weight=None, seed=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.row_subsample = row_subsample
        self.class_weight = class_weight
        self.seed = seed

    def fit(self, X, y, feature_names):
        self.feature_names_ = np.array(feature_names)
        self.trees_ = []
        self.feature_importances_ = np.zeros(X.shape[1])
        rng = np.random.default_rng(self.seed)
        n = X.shape[0]
        sample_size = int(n * self.row_subsample)
        max_features = max(3, int(np.sqrt(X.shape[1])))
        for i in range(self.n_estimators):
            idx = rng.choice(n, size=sample_size, replace=True)
            tree = FastRandomTree(
                max_depth=self.max_depth,
                min_leaf=90,
                max_features=max_features,
                n_thresholds=7,
                class_weight=self.class_weight,
                seed=self.seed + i + 1,
            )
            tree.fit(X[idx], y[idx], X.shape[1])
            self.trees_.append(tree)
            self.feature_importances_ += tree.feature_importances_
        total = self.feature_importances_.sum()
        if total:
            self.feature_importances_ /= total
        return self

    def predict_proba(self, X):
        probs = np.zeros((X.shape[0], 3))
        for tree in self.trees_:
            probs += tree.predict_proba(X)
        return probs / len(self.trees_)


def predict_with_threshold(proba, threshold):
    base = np.argmax(proba, axis=1)
    non_high = np.argmax(proba[:, :2], axis=1)
    return np.where(proba[:, 2] >= threshold, 2, non_high).astype(int)

def threshold_tradeoff(y_true, proba):
    rows = []
    for threshold in np.round(np.arange(0.20, 0.82, 0.02), 2):
        pred = predict_with_threshold(proba, threshold)
        _, _, metrics = classification_report_np(y_true, pred)
        rows.append({"threshold": threshold, **metrics})
    return pd.DataFrame(rows)

def select_threshold(tradeoff, min_recall=0.68):
    candidates = tradeoff[tradeoff["high_risk_recall"] >= min_recall].copy()
    if len(candidates):
        best = candidates.sort_values(["false_alarm_rate", "high_risk_precision"], ascending=[True, False]).iloc[0]
    else:
        best = tradeoff.sort_values(["high_risk_recall", "false_alarm_rate"], ascending=[False, True]).iloc[0]
    return float(best["threshold"])

In [14]:
def make_predictions_frame(split, ids, y_true, proba, threshold):
    pred = predict_with_threshold(proba, threshold)
    out = ids.copy()
    out["split"] = split
    out["actual_risk_label_num"] = y_true
    out["actual_risk_label"] = [LABEL_NAMES[int(v)] for v in y_true]
    out["predicted_risk_label_num"] = pred
    out["predicted_risk_label"] = [LABEL_NAMES[int(v)] for v in pred]
    out["prob_low_risk"] = proba[:, 0]
    out["prob_medium_risk"] = proba[:, 1]
    out["high_risk_probability"] = proba[:, 2]
    out["risk_score"] = (100 * out["high_risk_probability"]).round(1)
    out["is_high_risk_alert"] = (out["predicted_risk_label_num"] == 2).astype(int)
    out["is_false_alarm"] = ((out["predicted_risk_label_num"] == 2) & (out["actual_risk_label_num"] != 2)).astype(int)
    out["is_missed_high_risk"] = ((out["predicted_risk_label_num"] != 2) & (out["actual_risk_label_num"] == 2)).astype(int)
    return out

In [ ]:
# Business-readable local risk drivers for dashboard users
def build_driver_string(row):
    drivers = []

    if row.get("risk_factor_under_trend", 0) == 1:
        drivers.append("Sales below trend")
    if row.get("risk_factor_volatility", 0) == 1:
        drivers.append("High sales volatility")
    if row.get("risk_factor_sales_drop", 0) == 1:
        drivers.append("Recent sales drop")
    if row.get("risk_factor_customer_drop", 0) == 1:
        drivers.append("Customer drop")
    if row.get("stockout_flag", 0) == 1 or row.get("supply_pressure_score", 0) > 0.35:
        drivers.append("Supply pressure")
    if row.get("lead_time_risk_flag", 0) == 1:
        drivers.append("Long lead time")
    if row.get("unit_share_change_pct", 0) < -0.20:
        drivers.append("Losing category share")
    if row.get("customer_count_growth_pct", 0) < -0.20:
        drivers.append("Declining customer base")
    if row.get("market_tailwind_score", 0) < -8:
        drivers.append("Weak market context")
    if row.get("high_competition_flag", 0) == 1:
        drivers.append("High competitor pressure")
    if row.get("margin_gap_vs_target", 0) < -0.10:
        drivers.append("Margin below target")
    if row.get("campaign_active_with_no_sales", 0) == 1:
        drivers.append("Campaign not converting")

    return "; ".join(drivers[:5]) if drivers else "No major risk driver"

def attach_business_context(predictions, human):
    context_cols = [
        "product_id", "region_id", "year_month", "product_line", "product_category", "product_name",
        "lifecycle_stage", "units_sold", "revenue", "unique_customers", "avg_discount_pct",
        "market_demand_index", "competitor_pressure_index", "stockout_flag", "backorder_units",
        "campaign_flag", "campaign_spend", "website_visits", "demo_requests",
        "estimated_gross_profit", "estimated_gross_margin_pct", "product_unit_share_category_region",
        "unit_share_change_pct", "risk_factor_under_trend", "risk_factor_volatility",
        "risk_factor_sales_drop", "risk_factor_customer_drop", "supply_pressure_score",
        "lead_time_risk_flag", "customer_count_growth_pct", "market_tailwind_score",
        "high_competition_flag", "margin_gap_vs_target", "campaign_active_with_no_sales"]
    available = [c for c in context_cols if c in human.columns]
    out = predictions.merge(human[available], on=["product_id", "region_id", "year_month"], how="left")
    out["risk_drivers"] = out.apply(build_driver_string, axis=1)
    return out

In [ ]:
def create_tradeoff_chart(tradeoff):
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(tradeoff["threshold"], tradeoff["high_risk_recall"], label="High-Risk Recall", linewidth=2)
        ax.plot(tradeoff["threshold"], tradeoff["false_alarm_rate"], label="False Alarm Rate", linewidth=2)
        ax.plot(tradeoff["threshold"], tradeoff["high_risk_precision"], label="High-Risk Precision", linewidth=2)
        ax.set_title("High-Risk Threshold Trade-off")
        ax.set_xlabel("High-Risk Probability Threshold")
        ax.set_ylabel("Metric")
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        chart_path = OUTPUT_DIR / "false_alarm_recall_tradeoff.png"
        fig.savefig(chart_path, dpi=160)
        plt.close(fig)
        return str(chart_path)
    except Exception as exc:
        print(f"Chart export skipped: {exc}")
        return create_tradeoff_svg(tradeoff)

# Create a dependency-free SVG trade-off chart
def create_tradeoff_svg(tradeoff):
    width, height = 900, 520
    left, right, top, bottom = 80, 30, 50, 70
    plot_w = width - left - right
    plot_h = height - top - bottom

    x_min = float(tradeoff["threshold"].min())
    x_max = float(tradeoff["threshold"].max())

    def sx(x): return left + (float(x) - x_min) / (x_max - x_min) * plot_w

    def sy(y): return top + (1 - float(y)) * plot_h

    def points(col): return " ".join(f"{sx(x):.1f},{sy(y):.1f}" for x, y in zip(tradeoff["threshold"], tradeoff[col]))

    grid = []
    for i in range(6):
        y_val = i / 5
        y = sy(y_val)
        grid.append(f'<line x1="{left}" y1="{y:.1f}" x2="{width-right}" y2="{y:.1f}" stroke="#e5e7eb" stroke-width="1"/>')
        grid.append(f'<text x="{left-12}" y="{y+4:.1f}" text-anchor="end" font-size="12" fill="#374151">{y_val:.1f}</text>')

    for x_val in np.linspace(x_min, x_max, 7):
        x = sx(x_val)
        grid.append(f'<line x1="{x:.1f}" y1="{top}" x2="{x:.1f}" y2="{height-bottom}" stroke="#f3f4f6" stroke-width="1"/>')
        grid.append(f'<text x="{x:.1f}" y="{height-bottom+24}" text-anchor="middle" font-size="12" fill="#374151">{x_val:.2f}</text>')

    svg = f"""<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">
<rect width="100%" height="100%" fill="white"/>
<text x="{left}" y="28" font-size="22" font-weight="700" fill="#111827">High-Risk Threshold Trade-off</text>
{''.join(grid)}
<line x1="{left}" y1="{top}" x2="{left}" y2="{height-bottom}" stroke="#111827" stroke-width="1.5"/>
<line x1="{left}" y1="{height-bottom}" x2="{width-right}" y2="{height-bottom}" stroke="#111827" stroke-width="1.5"/>
<polyline fill="none" stroke="#2563eb" stroke-width="3" points="{points('high_risk_recall')}"/>
<polyline fill="none" stroke="#dc2626" stroke-width="3" points="{points('false_alarm_rate')}"/>
<polyline fill="none" stroke="#059669" stroke-width="3" points="{points('high_risk_precision')}"/>
<text x="{width-260}" y="78" font-size="14" fill="#2563eb">High-Risk Recall</text>
<text x="{width-260}" y="102" font-size="14" fill="#dc2626">False Alarm Rate</text>
<text x="{width-260}" y="126" font-size="14" fill="#059669">High-Risk Precision</text>
<text x="{width/2}" y="{height-22}" text-anchor="middle" font-size="14" fill="#111827">High-Risk Probability Threshold</text>
<text x="24" y="{height/2}" transform="rotate(-90 24 {height/2})" text-anchor="middle" font-size="14" fill="#111827">Metric</text>
</svg>"""
    chart_path = OUTPUT_DIR / "false_alarm_recall_tradeoff.svg"
    chart_path.write_text(svg, encoding="utf-8")
    return str(chart_path)


In [17]:
def run_interpretation_pipeline():
    print("Loading data...")
    data, human = load_inputs()

    X_train, y_train = split_xy(data["train_ml"])
    X_valid, y_valid = split_xy(data["valid_ml"])
    X_test, y_test = split_xy(data["test_ml"])

    fit_df = data["train_ml"].groupby(TARGET_COL, group_keys=False).sample(frac=0.65, random_state=RANDOM_SEED)
    X_fit, y_fit = split_xy(fit_df)

    counts = pd.Series(y_fit).value_counts().to_dict()
    max_count = max(counts.values())
    class_weight = {int(cls): max_count / count for cls, count in counts.items()}
    class_weight[2] = class_weight.get(2, 1.0) * 1.35

    print("Training compact Random Forest for row-level risk probabilities...")
    model = FastRandomForest(n_estimators=24, max_depth=7, row_subsample=0.72, class_weight=class_weight)
    model.fit(X_fit.to_numpy(dtype=np.float32), y_fit, X_fit.columns)

    valid_proba = model.predict_proba(X_valid.to_numpy(dtype=np.float32))
    tradeoff = threshold_tradeoff(y_valid, valid_proba)
    threshold = select_threshold(tradeoff, min_recall=0.68)

    print(f"Selected high-risk threshold: {threshold}")

    split_outputs = []
    for split, X, y in [
        ("train", X_train, y_train),
        ("valid", X_valid, y_valid),
        ("test", X_test, y_test)]:
        proba = model.predict_proba(X.to_numpy(dtype=np.float32))
        ids = data[f"{split}_ids"]
        split_outputs.append(make_predictions_frame(split, ids, y, proba, threshold))

    predictions = pd.concat(split_outputs, ignore_index=True)
    dashboard = attach_business_context(predictions, human)

    top_risk = (
        dashboard.sort_values(["high_risk_probability", "risk_score"], ascending=False)
        .head(250)
        .reset_index(drop=True))

    latest_month = dashboard["year_month"].max()
    latest_top_risk = (
        dashboard[dashboard["year_month"] == latest_month]
        .sort_values(["high_risk_probability", "risk_score"], ascending=False)
        .head(100)
        .reset_index(drop=True))

    summary = dashboard.groupby(["split", "predicted_risk_label"], as_index=False).agg(
        rows=("product_id", "count"),
        avg_high_risk_probability=("high_risk_probability", "mean"),
        avg_risk_score=("risk_score", "mean"),
        high_risk_alerts=("is_high_risk_alert", "sum"),
        false_alarms=("is_false_alarm", "sum"),
        missed_high_risks=("is_missed_high_risk", "sum"))

    model_metrics = []
    for split in ["train", "valid", "test"]:
        part = dashboard[dashboard["split"] == split]
        _, cm, metrics = classification_report_np(
            part["actual_risk_label_num"].to_numpy(),
            part["predicted_risk_label_num"].to_numpy())
        model_metrics.append({"split": split, **metrics})
        pd.DataFrame(
            cm,
            index=[f"actual_{LABEL_NAMES[i]}" for i in [0, 1, 2]],
            columns=[f"pred_{LABEL_NAMES[i]}" for i in [0, 1, 2]]
        ).to_csv(OUTPUT_DIR / f"{split}_confusion_matrix.csv")

    metrics_df = pd.DataFrame(model_metrics)
    importance = pd.DataFrame({
        "feature": model.feature_names_,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    chart_path = create_tradeoff_chart(tradeoff)

    dashboard.to_csv(OUTPUT_DIR / "risk_predictions_for_dashboard.csv", index=False)
    top_risk.to_csv(OUTPUT_DIR / "top_high_risk_products.csv", index=False)
    latest_top_risk.to_csv(OUTPUT_DIR / "latest_month_top_high_risk_products.csv", index=False)
    summary.to_csv(OUTPUT_DIR / "model_interpretation_summary.csv", index=False)
    tradeoff.to_csv(OUTPUT_DIR / "false_alarm_recall_tradeoff.csv", index=False)
    metrics_df.to_csv(OUTPUT_DIR / "risk_output_model_metrics.csv", index=False)
    importance.to_csv(OUTPUT_DIR / "risk_output_feature_importance.csv", index=False)

    report = {
        "selected_high_risk_threshold": threshold,
        "dashboard_rows": int(len(dashboard)),
        "latest_month": str(latest_month.date()),
        "chart_path": chart_path,
        "test_metrics": metrics_df[metrics_df["split"] == "test"].to_dict(orient="records"),
        "top_15_features": importance.head(15).to_dict(orient="records")}
    with open(OUTPUT_DIR / "model_interpretation_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)

    print("\nRisk output complete")
    print("-" * 80)
    print(f"Dashboard rows: {len(dashboard):,}")
    print(f"Selected threshold: {threshold}")
    print("\nModel metrics:")
    print(metrics_df.round(4).to_string(index=False))
    print("\nTop risk products:")
    print(top_risk[["product_id", "region_id", "year_month", "risk_score", "predicted_risk_label", "risk_drivers"]].head(10).to_string(index=False))
    print("\nTrade-off chart:", chart_path)
    print("Output files saved to:", OUTPUT_DIR.resolve())

    return {
        "dashboard": dashboard,
        "top_risk": top_risk,
        "tradeoff": tradeoff,
        "metrics": metrics_df,
        "importance": importance}

In [18]:
artifacts = run_interpretation_pipeline()

Loading data...
Training compact Random Forest for row-level risk probabilities...
Selected high-risk threshold: 0.58

Risk output complete
--------------------------------------------------------------------------------
Dashboard rows: 96,520
Selected threshold: 0.58

Model metrics:
split  accuracy  macro_f1  weighted_f1  high_risk_recall  high_risk_precision  false_alarm_rate  alert_rate    tp   fp   fn    tn
train    0.7209    0.7126       0.7243            0.7142               0.6756            0.1787      0.3621 15295 7345 6122 33758
valid    0.7095    0.6902       0.7156            0.6928               0.6796            0.1771      0.3584  4384 2067 1944  9605
 test    0.7031    0.6880       0.7083            0.6963               0.6555            0.1941      0.3681  3861 2029 1684  8426

Top risk products:
 product_id  region_id year_month  risk_score predicted_risk_label                                                         risk_drivers
       1061          2 2024-04-01      